# 00 — Veri Genel Bakışı

Konyaaltı imar verisinin yapısını, mahalle dağılımını ve pilot alan özelliklerini anlamak için. 

**Önkoşul:** `data/raw/konyaalti_imar_tum_veri.xlsx` dosyasının yerinde olması gerekiyor.

Çıktılar:
- `figures/00_kat_histogram.png`
- `figures/00_pilot_alan_dagilim.png`
- `tables/00_mahalle_kapsama.csv`
- `tables/00_pilot_alan_kapsama.csv`
- `data/processed/imar_full.gpkg`, `imar_pilot.gpkg`, `imar_pilot_known.gpkg`, `imar_pilot_unknown.gpkg`

In [ ]:
import sys
from pathlib import Path

# notebooks/ içinden çalıştığımız için kök dizini sys.path'e ekle
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point

from src.config import (
    DATA_PROCESSED, FIGURES, TABLES,
    CRS_PROJECTED, CRS_GEOGRAPHIC, PILOT_NEIGHBORHOODS,
    IMAR_XLSX, IMAR_SHEET,
)
from src.coord_utils import parse_coord_string

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

print("PROJECT_ROOT:", Path.cwd().parent)
print(f"Pilot mahalle sayısı: {len(PILOT_NEIGHBORHOODS)}")

In [ ]:
# --- Excel'i yükle ---
if not IMAR_XLSX.exists():
    raise FileNotFoundError(
        f"Veri dosyası bulunamadı: {IMAR_XLSX}\n"
        "konyaalti_imar_tum_veri.xlsx dosyasını data/raw/ klasörüne kopyalayın."
    )

df = pd.read_excel(IMAR_XLSX, sheet_name=IMAR_SHEET)
print(f"Shape: {df.shape}")
print(f"\nKolonlar: {list(df.columns)}")
print(f"\ndtypes:\n{df.dtypes}")
print(f"\nNaN sayısı (kolon başına):\n{df.isna().sum()}")
df.head()

In [ ]:
# --- kat_adedi temizliği ve histogram ---
df["kat_numeric"] = pd.to_numeric(
    df["kat_adedi"].replace({"-": np.nan, "": np.nan}),
    errors="coerce",
)

n_total = len(df)
n_known = int(df["kat_numeric"].notna().sum())
coverage_pct = 100 * n_known / n_total
print(f"Toplam kayıt: {n_total:,}")
print(f"Kat verisi olan: {n_known:,} (%{coverage_pct:.1f})")
print(f"Eksik: {n_total - n_known:,}")
print(f"\nKat dağılımı özeti:\n{df['kat_numeric'].describe()}")

FIGURES.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(9, 5))
max_kat = int(df["kat_numeric"].dropna().max())
ax.hist(
    df["kat_numeric"].dropna().astype(int),
    bins=range(1, max_kat + 2),
    edgecolor="black",
    color="#2E86AB",
)
ax.set_xlabel("Kat sayısı")
ax.set_ylabel("Kayıt sayısı")
ax.set_title(f"Konyaaltı imar — kat sayısı dağılımı (n={n_known:,})")
plt.tight_layout()
plt.savefig(FIGURES / "00_kat_histogram.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- Mahalle bazlı kapsama tablosu ---
mahalle_summary = (
    df.groupby("mahalle")
      .agg(
          toplam_kayit=("mahalle", "size"),
          kat_verili=("kat_numeric", "count"),
          ortalama_kat=("kat_numeric", "mean"),
          max_kat=("kat_numeric", "max"),
      )
      .assign(kapsama_pct=lambda x: 100 * x["kat_verili"] / x["toplam_kayit"])
      .sort_values("toplam_kayit", ascending=False)
      .round(2)
)
print(f"Toplam mahalle: {len(mahalle_summary)}")

TABLES.mkdir(parents=True, exist_ok=True)
mahalle_summary.to_csv(TABLES / "00_mahalle_kapsama.csv", encoding="utf-8-sig")
mahalle_summary

In [ ]:
# --- Pilot alan filtresi (15 mahalle) ---
df_pilot = df[df["mahalle"].isin(PILOT_NEIGHBORHOODS)].copy()
n_pilot = len(df_pilot)
n_pilot_known = int(df_pilot["kat_numeric"].notna().sum())
n_pilot_unknown = n_pilot - n_pilot_known

print(f"Pilot alandaki kayıt: {n_pilot:,}")
print(f"Kat verili: {n_pilot_known:,} (%{100*n_pilot_known/n_pilot:.1f})")
print(f"Eksik: {n_pilot_unknown:,}")

found = sorted(df_pilot["mahalle"].unique())
missing = [m for m in PILOT_NEIGHBORHOODS if m not in found]
print(f"\nVeride bulunan pilot mahalle: {len(found)} / {len(PILOT_NEIGHBORHOODS)}")
if missing:
    print(f"⚠ Veride bulunamayan: {missing}")

In [ ]:
# --- DMS koordinatları parse et ---
parsed = df["cografi_koordinat"].apply(parse_coord_string)
df["lat"] = parsed.apply(lambda x: x[0] if x is not None else np.nan)
df["lon"] = parsed.apply(lambda x: x[1] if x is not None else np.nan)

n_parsed = int(df["lat"].notna().sum())
print(f"Başarılı parse: {n_parsed:,} / {len(df):,} (%{100*n_parsed/len(df):.1f})")
print(f"Başarısız: {len(df) - n_parsed:,}")
if n_parsed > 0:
    print(f"\nlat aralığı: [{df['lat'].min():.4f}, {df['lat'].max():.4f}]")
    print(f"lon aralığı: [{df['lon'].min():.4f}, {df['lon'].max():.4f}]")

In [ ]:
# --- GeoDataFrame: EPSG:4326 → EPSG:32636 ---
df_geo = df.dropna(subset=["lat", "lon"]).copy()
gdf = gpd.GeoDataFrame(
    df_geo,
    geometry=gpd.points_from_xy(df_geo["lon"], df_geo["lat"]),
    crs=CRS_GEOGRAPHIC,
).to_crs(CRS_PROJECTED)
print(f"GeoDataFrame: {len(gdf):,} geçerli geometri, CRS = {gdf.crs}")

gdf_pilot = gdf[gdf["mahalle"].isin(PILOT_NEIGHBORHOODS)].copy()
gdf_pilot_known = gdf_pilot[gdf_pilot["kat_numeric"].notna()].copy()
gdf_pilot_unknown = gdf_pilot[gdf_pilot["kat_numeric"].isna()].copy()
print(
    f"Pilot alan: {len(gdf_pilot):,} | "
    f"bilinen: {len(gdf_pilot_known):,} | "
    f"eksik: {len(gdf_pilot_unknown):,}"
)

In [ ]:
# --- Mekânsal görselleştirme ---
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Sol — tüm Konyaaltı, mahalle bazlı renklendirme
gdf.plot(
    column="mahalle", categorical=True, legend=False,
    markersize=2, ax=axes[0], cmap="tab20",
)
axes[0].set_title(f"Konyaaltı tamamı (n={len(gdf):,})")
axes[0].set_aspect("equal")
axes[0].ticklabel_format(style="plain")

# Sağ — pilot alan: kat verisi var/yok ayrımı
if len(gdf_pilot_unknown) > 0:
    gdf_pilot_unknown.plot(
        ax=axes[1], color="#E63946", markersize=4,
        label=f"Eksik kat (n={len(gdf_pilot_unknown):,})", alpha=0.6,
    )
gdf_pilot_known.plot(
    ax=axes[1], color="#2A9D8F", markersize=4,
    label=f"Kat verili (n={len(gdf_pilot_known):,})", alpha=0.6,
)
axes[1].set_title(f"Pilot alan (n={len(gdf_pilot):,})")
axes[1].legend(loc="upper right")
axes[1].set_aspect("equal")
axes[1].ticklabel_format(style="plain")

plt.tight_layout()
plt.savefig(FIGURES / "00_pilot_alan_dagilim.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- Pilot alan tabakalı özet ---
pilot_summary = (
    gdf_pilot.groupby("mahalle")
       .agg(
           kayit=("mahalle", "size"),
           kat_verili=("kat_numeric", "count"),
           ortalama_kat=("kat_numeric", "mean"),
           max_kat=("kat_numeric", "max"),
       )
       .assign(kapsama_pct=lambda x: 100 * x["kat_verili"] / x["kayit"])
       .reindex(PILOT_NEIGHBORHOODS)
       .round(2)
)
pilot_summary.to_csv(TABLES / "00_pilot_alan_kapsama.csv", encoding="utf-8-sig")
pilot_summary

In [ ]:
# --- GeoPackage'a kaydet ---
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# DMS string kolonu drop et — geometri zaten temsil ediyor
drop_cols = [c for c in ["cografi_koordinat"] if c in gdf.columns]

gdf.drop(columns=drop_cols).to_file(DATA_PROCESSED / "imar_full.gpkg", driver="GPKG")
gdf_pilot.drop(columns=drop_cols).to_file(DATA_PROCESSED / "imar_pilot.gpkg", driver="GPKG")
gdf_pilot_known.drop(columns=drop_cols).to_file(DATA_PROCESSED / "imar_pilot_known.gpkg", driver="GPKG")
gdf_pilot_unknown.drop(columns=drop_cols).to_file(DATA_PROCESSED / "imar_pilot_unknown.gpkg", driver="GPKG")

print("Yazılan dosyalar:")
for f in sorted(DATA_PROCESSED.glob("imar_*.gpkg")):
    print(f"  {f.name} — {f.stat().st_size/1024:.1f} KB")

## Özet

- Toplam ve pilot alan veri kapsaması yukarıda raporlandı.
- DMS koordinatlar EPSG:32636'ya projekte edildi.
- 4 ayrı GeoPackage `data/processed/` altına yazıldı.

## Sonraki Adım

`01_grid_setup.ipynb` — Konyaaltı pilot alan sınırını OSM'den çekip 30 m + 100 m grid sistemini kuracağız.